# ImageNet Training Notebook (Kaggle)

Train FracBNN / Ada-FracBNN on ImageNet with three model variants:

| model\_id | Architecture | Description |
|-----------|-------------|-------------|
| 0 | `binput-pg-quant-shortcut` | Baseline PG |
| 1 | `adaptive-pg` | Adaptive PG (Ada-FracBNN) |
| 2 | `adaptive-pg-kd` | Adaptive PG + Knowledge Distillation |

## Before you start
1. **Add the dataset**: Click **Add data** (right panel) and search for `imagenet-object-localization-challenge`. Accept the competition rules if prompted.
2. **Enable GPU**: Settings > Accelerator > **GPU T4 x2** (or P100).
3. **Enable Internet**: Settings > Internet > **On** (needed for git clone and pip install).
4. **Persistence**: Set Settings > Persistence > **Files only** so `/kaggle/working/` survives restarts.

### Kaggle paths
- `/kaggle/input/` -- read-only attached datasets (ImageNet train is already in ImageFolder format here)
- `/kaggle/working/` -- writable output directory (~70 GB); checkpoints go here
- Session limit: 12 h with GPU

## Cell 1 -- GPU and Disk Check

In [ ]:
!nvidia-smi
import torch, os
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"VRAM: {mem_gb:.1f} GB")
    print(f"GPU count: {torch.cuda.device_count()}")
!df -h /kaggle/working

## Cell 2 -- Clone Repo

In [ ]:
import os

REPO_DIR = "/kaggle/working/endingengineering"
REPO_URL = "https://github.com/YOUR_USERNAME/endingengineering.git"  # <-- replace
BRANCH   = "feat/imagenet-colab-bootstrap"

if not os.path.exists(REPO_DIR):
    !git clone --branch $BRANCH $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git fetch origin $BRANCH && git checkout $BRANCH && git pull origin $BRANCH

os.chdir(REPO_DIR)
!git log --oneline -5

## Cell 3 -- Install Dependencies

In [ ]:
!pip install -q -r requirements.txt

## Cell 4 -- Smoke Test (verify ImageNet models load)

In [ ]:
!python test_imagenet_models.py

## Cell 5 -- Prepare ImageNet Data

Kaggle's ImageNet `train/` is already in ImageFolder format (class sub-dirs) under `/kaggle/input/`.
The `val/` folder is **flat** (all 50 000 images in one directory), so we reorganise it into class sub-dirs
inside `/kaggle/working/` using the competition's `LOC_val_solution.csv`.

We then create a unified `IMAGENET_DIR` with symlinks so `imagenet.py` sees `train/` and `val/`.

In [ ]:
import csv, shutil, pathlib, os

KAGGLE_DS   = "/kaggle/input/imagenet-object-localization-challenge"
IMAGENET_DIR = "/kaggle/working/imagenet"

train_src = os.path.join(KAGGLE_DS, "ILSVRC", "Data", "CLS-LOC", "train")
val_src   = os.path.join(KAGGLE_DS, "ILSVRC", "Data", "CLS-LOC", "val")
sol_csv   = os.path.join(KAGGLE_DS, "LOC_val_solution.csv")

# --- symlink train (read-only is fine for DataLoader) ---
train_dst = os.path.join(IMAGENET_DIR, "train")
os.makedirs(IMAGENET_DIR, exist_ok=True)
if not os.path.exists(train_dst):
    os.symlink(train_src, train_dst)
    print(f"Symlinked train -> {train_src}")
else:
    print(f"train/ already exists at {train_dst}")

# --- reorganise val into class sub-dirs ---
val_dst = pathlib.Path(IMAGENET_DIR) / "val"
marker  = val_dst / ".reorganised"

if marker.exists():
    print("val/ already reorganised -- skipping")
else:
    print("Reorganising val/ into class sub-dirs (takes ~2 min) ...")
    val_dst.mkdir(parents=True, exist_ok=True)
    moved = 0
    with open(sol_csv) as f:
        for row in csv.DictReader(f):
            img_id = row["ImageId"]
            wnid   = row["PredictionString"].split()[0]
            cls_dir = val_dst / wnid
            cls_dir.mkdir(exist_ok=True)
            src = pathlib.Path(val_src) / f"{img_id}.JPEG"
            dst = cls_dir / f"{img_id}.JPEG"
            if src.exists() and not dst.exists():
                shutil.copy2(str(src), str(dst))
                moved += 1
    marker.touch()
    print(f"Done -- copied {moved} images into class sub-dirs")

# --- verify ---
train_classes = [d for d in os.listdir(train_dst) if os.path.isdir(os.path.join(train_dst, d))]
val_classes   = [d for d in os.listdir(val_dst)   if os.path.isdir(os.path.join(str(val_dst), d))]
print(f"Train classes: {len(train_classes)},  Val classes: {len(val_classes)}")
!du -sh $IMAGENET_DIR/val

## Cell 6 -- Set Checkpoint Directory

Everything under `/kaggle/working/` is saved as notebook output and persists across sessions
(when Persistence is set to **Files only**).

In [ ]:
SAVE_DIR = "/kaggle/working/checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Checkpoints will be saved to {SAVE_DIR}")

---
## Cell 7 -- Train Model 0: Baseline PG (`binput-pg-quant-shortcut`)

Full schedule: 120 epochs.  
Kaggle GPU sessions last 12 h -- use Cell 11 to resume across sessions.  
Reduce `-e` for a shorter run.

In [ ]:
os.chdir(REPO_DIR)

!python imagenet.py \
    -id 0 \
    -e  120 \
    -b  128 \
    -lr 5e-4 \
    -wd 1e-5 \
    -d  $IMAGENET_DIR \
    --label_smoothing 0.1 \
    -gpu 0 \
    -s

!cp -a save_ImageNet_model "$SAVE_DIR/save_ImageNet_model_baseline"

## Cell 8 -- Train Model 1: Adaptive PG (`adaptive-pg`)

In [ ]:
!python imagenet.py \
    -id 1 \
    -e  120 \
    -b  128 \
    -lr 5e-4 \
    -wd 1e-5 \
    -d  $IMAGENET_DIR \
    -ts 0.15 \
    -sw 0.01 \
    --label_smoothing 0.1 \
    -gpu 0 \
    -s

!cp -a save_ImageNet_model "$SAVE_DIR/save_ImageNet_model_adaptive"

## Cell 9 -- Train Model 2: Adaptive PG + KD (`adaptive-pg-kd`)

Uses a **pretrained torchvision ResNet-50** as teacher (downloaded automatically).  
Batch size is **64** because both student and teacher must fit in GPU memory.

In [ ]:
!python imagenet.py \
    -id 2 \
    -e  120 \
    -b  64 \
    -lr 5e-4 \
    -wd 1e-5 \
    -d  $IMAGENET_DIR \
    -ts 0.15 \
    -sw 0.01 \
    -temp  4.0 \
    -alpha 0.7 \
    --teacher_arch resnet50 \
    --teacher_pretrained \
    --label_smoothing 0.1 \
    -gpu 0 \
    -s

!cp -a save_ImageNet_model "$SAVE_DIR/save_ImageNet_model_kd"

## Cell 10 -- Evaluate All Saved Models

In [ ]:
import glob

variants = [
    ("Baseline PG",      0, f"{SAVE_DIR}/save_ImageNet_model_baseline"),
    ("Adaptive PG",      1, f"{SAVE_DIR}/save_ImageNet_model_adaptive"),
    ("Adaptive PG + KD", 2, f"{SAVE_DIR}/save_ImageNet_model_kd"),
]

for name, mid, path in variants:
    ckpts = sorted(glob.glob(os.path.join(path, "model_*.pt")))
    if not ckpts:
        print(f"\n[SKIP] {name} -- no checkpoint found in {path}")
        continue
    ckpt = ckpts[0]
    print(f"\n{'='*60}")
    print(f"Evaluating: {name}  (checkpoint: {ckpt})")
    print(f"{'='*60}")
    !python imagenet.py -id $mid -t -r "$ckpt" -d $IMAGENET_DIR -b 128 -gpu 0

## Cell 11 -- Resume Training After Session Timeout

Kaggle GPU sessions last up to 12 h.  
Re-run Cells 1-6 (fast -- repo is cached, val is already reorganised), then use this cell.  
Edit the three variables below to match the variant you were training.

In [ ]:
# --- edit these three lines to match your variant ---
MODEL_ID   = 1
CKPT_MODEL = f"{SAVE_DIR}/save_ImageNet_model_adaptive/model_adaptive-pg.pt"
CKPT_STATE = f"{SAVE_DIR}/save_ImageNet_model_adaptive/states_adaptive-pg.pt"
# ----------------------------------------------------

os.chdir(REPO_DIR)

!python imagenet.py \
    -id $MODEL_ID \
    -e  120 \
    -b  128 \
    -lr 5e-4 \
    -wd 1e-5 \
    -d  $IMAGENET_DIR \
    -r  "$CKPT_MODEL" \
    -l  "$CKPT_STATE" \
    -ts 0.15 \
    -sw 0.01 \
    --label_smoothing 0.1 \
    -gpu 0 \
    -s

---
## Reference

### Hyperparameters for good accuracy

| Parameter | Value | Notes |
|-----------|-------|-------|
| Epochs | 120 | Full schedule |
| Batch size | 128 (baseline/adaptive), 64 (KD) | P100/T4 safe |
| Learning rate | 5e-4, linear decay | |
| Weight decay | 1e-5 | |
| Label smoothing | 0.1 | |
| Target sparsity | 0.15 | 15% channels get 2-bit |
| Sparsity weight | 0.01 | |
| KD temperature | 4.0 | |
| KD alpha | 0.7 | |
| Teacher | ResNet-50 pretrained | torchvision weights |

### Expected accuracy (full ImageNet, 120 epochs)

| Variant | Top-1 |
|---------|-------|
| Baseline PG | ~71-72% (paper: 71.7%) |
| Adaptive PG | ~71.5-72.5% |
| Adaptive PG + KD | ~72-73% |

### Kaggle runtime notes

- GPU quota: 30 h/week (T4 x2) or 20 h/week (P100)
- Session limit: 12 h
- ~60-90 min per epoch on full ImageNet with a single GPU
- 120 epochs needs multiple sessions; use Cell 11 to resume
- `/kaggle/working/` persists across sessions when Persistence is enabled